In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.6 Batching and padding

> 🎯 **Goal:** Feed many sequences to the model at once, handling uneven lengths with padding and a mask so the fake tokens never corrupt training.

**Why this matters.** GPUs are fast because they do the same operation on many examples in parallel. To use that, we stack many sequences into one rectangular *batch* tensor. But a rectangle needs every row the same length, and real sentences are not. Solving this cleanly (and avoiding one infamous bug) is the last piece before training can begin.

**The intuition.** Imagine lining up sentences in a spreadsheet, one per row. A spreadsheet must have equal-length rows, so short sentences leave blank trailing cells. You fill those blanks with a placeholder (the *pad token*) just to square off the grid. But you must remember those cells are fake. The *mask* is a second grid marking which cells are real (True) and which are filler (False), so nothing downstream mistakes padding for content.

**The mechanics.** Two situations show up. When you slice fixed-size *windows* from one long token stream (`make_batch` with `block_size`), every window is the same length by construction, so no padding is needed; here `x` is the input and `y` is `x` shifted one position (the next-token target). When inputs are *ragged* (different lengths, like two sentences), `pad_batch` extends each to the longest length using `pad_id` and returns a `mask` marking real tokens. Later, attention and the loss read that mask and skip the pad positions entirely.

In [ ]:
from pocketlm import CharTokenizer, make_batch, pad_batch

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
x, y = make_batch(data, block_size=16, batch_size=4,
                  generator=torch.Generator().manual_seed(0))
print("x", tuple(x.shape), " y", tuple(y.shape), "(y is x shifted by one)")

padded, mask = pad_batch([tok.encode("hi"), tok.encode("hello")], pad_id=0)
print("padded:\n", padded)
print("mask (True = real token):\n", mask)

**What just happened.** `make_batch` produced `x` and `y`, both shaped `(batch_size, block_size)` = `(4, 16)`: four windows of sixteen tokens, with `y` being `x` nudged one step ahead so each position's label is the token that actually came next. Then `pad_batch` took two unequal sequences (`"hi"` and `"hello"`), padded the shorter one up to the longer length with `pad_id=0`, and returned a mask where True marks the real tokens and False marks the padding. The mask has the same shape as the padded batch, and the number of True values in the first row equals the real length of `"hi"`.

⚠️ **Common pitfalls**
- **Mask leakage** (the classic bug): if you forget to apply the mask, the model attends to and trains on the pad tokens, learning from positions that were never real text. The loss looks fine but the model is quietly corrupted. Always carry the mask alongside the padded batch and feed it to attention and the loss.
- Reusing a real token id as the pad id: if `pad_id` collides with an id that appears in actual text, you can no longer tell padding from content by value, and the mask becomes your only defense. Prefer a dedicated, reserved pad id.

🏋️ **Try it yourself.** Add a third, even longer sequence (like `tok.encode("greetings")`) to the `pad_batch` call and reprint `padded` and `mask`. Watch every row stretch to the new maximum length and the mask grow more False cells in the shorter rows. Then flip `pad_id` to a different value and confirm the mask, not the id, is what reliably identifies the padding.

In [ ]:
assert tuple(x.shape) == (4, 16)
assert padded.shape == mask.shape
assert int(mask[0].sum()) == len(tok.encode("hi"))